# Recherche image -> image

Notebook nettoy? depuis `filter_by_image (1).ipynb`. Il contient l'approche d?j? impl?ment?e avec InsightFace, FAISS et Qdrant pour filtrer par image ou par intersection/union de r?f?rences.

In [ ]:
import os

import cv2
import numpy as np
import pandas as pd
from sklearn.preprocessing import normalize

from collections import defaultdict, Counter

from concurrent.futures import ThreadPoolExecutor, as_completed

from insightface.app import FaceAnalysis

import faiss
from qdrant_client import QdrantClient, models


client = QdrantClient(url="http://localhost:6333")
app = FaceAnalysis(name=r"..\models\antelopev2", providers=['CPUExecutionProvider']) 


class FilterByImage:
    def __init__(self, folder_name: str, images_path: str, app, client, h = 640, w = 640, vec_dim = 512, max_workers=8):
        self.folder_name = folder_name
        self.col_name = f"faces_{self.folder_name}"
        self.images_path = FilterByImage._check_path(images_path) 

        self.client = client
        self.max_workers = max_workers
        self.vec_dim = vec_dim
        
        self.app = app
        self.app.prepare(ctx_id=0, det_size=(h, w)) 
        

        self._df = pd.DataFrame(columns=["image", "embeddings"])
        self._init_dataframe()
        self._create_clusters()
        self._init_vector_store()

        self.count_actions = 0 
        self.reference_count = self._upload_ref_count() # On met cette valeur à 10 sinon on aura jamais le clustering auto


    @property
    def dataframe(self):
        return self._df.copy()
    

    @dataframe.setter
    def dataframe(self, new_df):
        if type(new_df) == pd.core.frame.DataFrame:
            self._df = new_df


    def _upload_ref_count(self):
        return self._df.shape[0]* (not self._df.empty)+10*(self._df.empty)
    
    
    @staticmethod
    def _check_path(path: str):
        if not os.path.exists(path):
            raise FileNotFoundError(f"Path not found: '{path}'")

        if not os.path.isdir(path):
            raise NotADirectoryError(f"Path is not a directory: '{path}'")

        return path

        
    @staticmethod
    def _get_images_path(im_path: str):
        paths = []
        for dir, _, files_name in os.walk(im_path):
            paths.extend([dir+"\\"+file for file in files_name])
        return paths
    
    @staticmethod
    def _fast_imread(path):
        with open(path, "rb") as f:
            arr = np.frombuffer(f.read(), dtype=np.uint8)
        return cv2.imdecode(arr, cv2.IMREAD_COLOR)

    @staticmethod
    def _add_padding(image, padding_percent: float = 0.4):
        h, w = image.shape[:2]
        pad_h = int(h * padding_percent)
        pad_w = int(w * padding_percent)
        
        padded = cv2.copyMakeBorder(
            image, 
            pad_h, pad_h, pad_w, pad_w,
            cv2.BORDER_CONSTANT,
            value=[0, 0, 0]  
        )
        return padded

    def _process_image(self, path: str):
        img = FilterByImage._fast_imread(path)
       
        im_padded = FilterByImage._add_padding(img, padding_percent=0.4)
        faces = self.app.get(im_padded)
        
        results = []
        for face in faces:
            results.append((path, face.embedding))
        return results
      

    def _init_dataframe(self):
        paths = FilterByImage._get_images_path(self.images_path)
        data = defaultdict(list)
        with ThreadPoolExecutor(max_workers= self.max_workers) as executor:
            futures = [executor.submit(self._process_image, path) for path in paths]
            for future in as_completed(futures):
                for path, embedding in future.result():
                    data["image"].append(path)
                    data["embeddings"].append(embedding)

        self.dataframe = pd.DataFrame(data)


    def _init_vector_store(self):
        df = self.dataframe
        
        client.delete_collection(collection_name=self.col_name)
        
        client.create_collection(
        collection_name=self.col_name,
        vectors_config= models.VectorParams(size = self.vec_dim, distance = models.Distance.COSINE, on_disk=True))
        
        if not df.empty:
            points = [models.PointStruct(id= row[0], vector= row[1]["embeddings"], payload= {"image": row[1]["image"], "person_id": row[1]["person_id"]}) for row in df.iterrows()]

            client.upsert(
                collection_name= self.col_name,
                points = points
            )

    def _create_clusters(self):
        df = self.dataframe

        if not df.empty:

            df = df.loc[:, ["image", "embeddings"]].reset_index(drop = True)

            X = np.vstack(df["embeddings"].to_numpy())
            X_norm = normalize(X).astype('float32')

            n_samples, dim = X_norm.shape

            index = faiss.IndexFlatIP(dim)  
            index.add(X_norm)

            similarity_threshold = 0.6
            k_neighbors = 50

            D, I = index.search(X_norm, k_neighbors)

            edges = []
            for i in range(n_samples):
                for j, sim in zip(I[i], D[i]):
                    if i != j and sim >= similarity_threshold:
                        edges.append((i, j))

            parent = list(range(n_samples))

            def find(x):
                if parent[x] != x:
                    parent[x] = find(parent[x])
                return parent[x]

            def union(x, y):
                px, py = find(x), find(y)
                if px != py:
                    parent[px] = py


            for i, j in edges:
                union(i, j)


            cluster_map = {}
            labels = np.zeros(n_samples, dtype=int)
            current_cluster = 0

            for i in range(n_samples):
                root = find(i)
                if root not in cluster_map:
                    cluster_map[root] = current_cluster
                    current_cluster += 1
                labels[i] = cluster_map[root]


            df["person_id"] = labels
            
            df = df.drop_duplicates(subset=["image", "person_id"])
            self.dataframe = df

        else:
            self.dataframe = pd.DataFrame(columns=["image", "embeddings", "person_id"])

    @staticmethod
    def _check_image_path(path: str):
        if not os.path.isfile(path):
            raise FileNotFoundError(f"Path is not a file: {path}")

        if os.path.splitext(path)[1].lower() not in [".jpg", ".jpeg", ".png"]:
            raise ValueError(f"Unsupported file extension: {path}")

        return path
    
    
    def prediction(self, image_path: str):

        ipt_img = cv2.imread(FilterByImage._check_image_path(image_path))
        if ipt_img is None:
            return None
        
        ipt_faces = self.app.get(FilterByImage._add_padding(ipt_img))

        img_embed = ipt_faces[0].embedding

        res = client.query_points(
            collection_name= self.col_name,
            query = img_embed,
            limit = 50
            )
        return res 
    
    def filter_df(self, image_path: str, df = None):
        if df is None:
            df = self.dataframe

        res = self.prediction(image_path)

        threshold = 0.6
        D = [point for point in res.points if point.score >threshold]
        
        if D and isinstance(df, pd.core.frame.DataFrame):
            class_lst = [cla.payload["person_id"] for cla in D]

            pred = max(Counter(class_lst).items(), key = lambda x: x[1])[0]
            df_filter = df[df["person_id"] == pred]
            
            return df_filter.drop_duplicates(subset=["image", "person_id"])
        else:
            return []


    def filter_union(self, *paths):
        output = []
        for path in paths:
            if self._check_image_path(path):
                res = self.filter_df(path)["image"].drop_duplicates().tolist()
                output.extend(res)

        return set(output)
    

    def filter_intersection(self, *paths):
        df = self.dataframe
        for path in paths:
            if self._check_image_path(path):
                filter_images = self.filter_df(path, df)["image"].drop_duplicates().tolist()
                df = df[df["image"].isin(filter_images)]

        if isinstance(df, pd.core.frame.DataFrame):
            return df["image"].drop_duplicates().tolist()
        else:
            return []
        

    def _check_clustering(self, p: float = 0.5):
        
        if (self.count_actions > p* self.reference_count) and self.reference_count>=10: # Condition à optimiser
            return True
        else:
            return False

    def _reset_clusters(self):
        self._create_clusters()
        self._init_vector_store()
        self.count_actions = 0
        self.reference_count = self._upload_ref_count()


    def add_image(self, path):
        if self._check_image_path(path):

            df = self.dataframe
            load_img = self._fast_imread(path)
            faces = self.app.get(load_img)
            self.count_actions+=len(faces)
            
            if self._check_clustering():
                for face in faces:
                    df.loc[df.shape[0]] = [path, face.embedding, 0] 
                    
                self.dataframe = df
                print("RESET CLUSTERING...")
                self._reset_clusters()
                
            else:
                points_lst = []
                for face in faces:
                    res = self.client.query_points(collection_name= self.col_name, query = face.embedding, limit = 50)
                    threshold = 0.6
                    D = [point for point in res.points if point.score >threshold]
                    print(res)
                    if D:
                        
                        class_lst = [cla.payload["person_id"] for cla in D]
                        pred = max(Counter(class_lst).items(), key = lambda x: x[1])[0]
                    else:
                        if df.empty:
                            pred = 0
                        else:
                            pred = max(df["person_id"])+1
                            
        
                    # df.loc[df.shape[0]] = [path, face.embedding, pred]
                    new_row = pd.DataFrame({
                        'image': [path],
                        'embeddings': [face.embedding],  
                        'person_id': [pred]
                    })
                    df = pd.concat([df, new_row], ignore_index=True)
                    # points_lst.append(models.PointStruct(id= max(df.index)+1, vector= face.embedding, payload= {"image": path, "person_id": pred}))
                    client.upsert(collection_name= self.col_name, points=[models.PointStruct(id= max(df.index)+1, vector= face.embedding, payload= {"image": path, "person_id": pred})])
                
                # client.upsert(collection_name= self.col_name, points=points_lst)
                self.dataframe = df


    def add_images(self, *paths):
        for path in paths:
            if self._check_image_path(path):
                self.add_image(path)
        print(f"The following images have been added to the folder '{self.folder_name}': "+ ", ".join(paths))


    def remove_images(self, *paths):
        df = self.dataframe
        for path in paths:
            if self._check_image_path(path):
                del_index = df[df["image"] == path].index.tolist()
                if del_index:
                    df = df.drop(index= del_index)
                    self.client.delete(collection_name= self.col_name, points_selector= del_index)
                    print(f"The following image has been removed from the folder '{self.folder_name}' : "+ ", ".join(paths))
                    
        self.dataframe = df  
        self.reference_count = self._upload_ref_count()



# Dossier 1: 

 
# Dossier: Voici l'objet associé à ton dossier image
# Tu dois remplacer images_path par le chemin de ton dossier image.
# Pour le folder_name, tu peux laisser comme ça ou donner un autre nom (dans le cas où tu voudrais tester avec d'autres dossier image, 
# il faudrait créer une nouvelle instance de cette classe)
dos1 = FilterByImage(folder_name= "folder1", images_path = r"CHEMIN VERS LE DOSSIER IMAGE", app= app, client = client)

In [ ]:
# Les fonctions à tester

# dos1.filter_df(CHEMIN VERS UNE IMAGE) : Pour filtrer par rapport à une image
# dos1.filter_union(CHEMIN1, CHEMIN2, etc...) : Pour filtrer par rapport à plusieurs images en mode UNION(on prend toutes les images qui contiennent les personnes présentes dans le filtre.[Pas besoin d'avoir toutes les personnes du filtre dans la même image])
# dos1.filter_intersection(CHEMIN1, CHEMIN2, etc...): Pour filtrer par rapport à plusieurs images, en mode INTERSECTION (même principe que pour UNION, mais là, toutes les personnes du filtre doivent se trouver sur chaque image)
# dos1.add_images(CHEMIN1, CHEMIN2, etc...: Pour ajouter une ou plusieurs images au dossier 
# dos1.remove_images(CHEMIN1, CHEMIN2, etc...) Pour retirer un ou plusieurs images du dossier

# PS1: Si tu utilises filter_union ou filter_intersection avec une seule image, ça va te faire un filtrage normal :-)
# PS2: pour ragarder le dataframe tu peux : print(dos1.dataframe)